In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/somukavyasri/protein-bilstm-tokenizer/keras/default/1/bilstm_model.h5
/kaggle/input/models/somukavyasri/protein-bilstm-tokenizer/keras/default/1/tokenizer.pkl


In [2]:
import tensorflow as tf
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences

# IMPORTANT: adjust path according to your uploaded dataset name
MODEL_PATH = "/kaggle/input/models/somukavyasri/protein-bilstm-tokenizer/keras/default/1/bilstm_model.h5"
TOKENIZER_PATH = "/kaggle/input/models/somukavyasri/protein-bilstm-tokenizer/keras/default/1/tokenizer.pkl"

model = tf.keras.models.load_model(MODEL_PATH)

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

max_len = 600

print("Model and tokenizer loaded successfully.")

2026-03-05 07:05:45.502241: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772694345.683384      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772694345.736199      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772694346.164046      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772694346.164082      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772694346.164085      24 computation_placer.cc:177] computation placer alr

Model and tokenizer loaded successfully.


In [3]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 600

def predict_sequence(sequence):
    
    sequence = sequence.upper().strip()
    
    seq_encoded = tokenizer.texts_to_sequences([sequence])
    seq_pad = pad_sequences(seq_encoded,
                            maxlen=max_len,
                            padding='post',
                            truncating='post')
    
    prob = float(model.predict(seq_pad)[0][0])
    label = 1 if prob > 0.5 else 0
    
    if prob > 0.8:
        confidence = "High Confidence"
    elif prob > 0.6:
        confidence = "Moderate Confidence"
    else:
        confidence = "Low Confidence"
    
    return {
        "Probability": round(prob, 4),
        "Prediction": "Likely Crystallizable" if label == 1 
                      else "Likely Non-Crystallizable",
        "Confidence": confidence
    }

In [4]:
import requests

def get_seq(uniprot_id):
    fasta_url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    r = requests.get(fasta_url, headers={"Accept": "text/plain"})
    r.raise_for_status()
    # Extract sequence lines (skip header)
    seq = "".join(line.strip() for line in r.text.splitlines() if not line.startswith(">"))
    return seq[:600]

id_name=input("enter protein_id:")
print("sequence:",get_seq(id_name))

StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.

# Alpha-synuclein (associated with Parkinson's disease)

In [ ]:
sequence_input = input("Enter protein sequence: ")

result = predict_sequence(sequence_input)

print("\n=== Prediction Result ===")
for key, value in result.items():
    print(f"{key}: {value}")

# Hen Egg-White Lysozyme (HEWL)

In [ ]:
sequence_input = input("Enter protein sequence: ")

result = predict_sequence(sequence_input)

print("\n=== Prediction Result ===")
for key, value in result.items():
    print(f"{key}: {value}")

# Bovine Pancreatic Trypsin Inhibitor (BPTI)

In [ ]:
sequence_input = input("Enter protein sequence: ")

result = predict_sequence(sequence_input)

print("\n=== Prediction Result ===")
for key, value in result.items():
    print(f"{key}: {value}")

# Maltose-Binding Protein (MalE) Sequence (E. coli)

In [ ]:
sequence_input = input("Enter protein sequence: ")

result = predict_sequence(sequence_input)

print("\n=== Prediction Result ===")
for key, value in result.items():
    print(f"{key}: {value}")